### 1. What factors affect the probability a basketball team wins (during the regular season)?

In [2]:
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
import time

In [1]:
## Using data from 21-26 seasons, spanning 82 * 6 = 496 games ##
seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

# Getting seasons data and building the dataset
dfs_base = []

for season in seasons:
    print(f"Fetching {season}...")

    data = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame",
        measure_type_detailed_defense="Base",
        timeout=60
    )

    df_season = data.get_data_frames()[0]
    df_season["SEASON"] = season
    dfs_base.append(df_season)
    time.sleep(2)

df_all = pd.concat(dfs_base, ignore_index=True)

## Building the model dataset ##
# Target
target = "W_PCT"

# Columns to remove from features
drop_exact = {
    # identifiers / labels
    "TEAM_ID",
    "TEAM_NAME",
    "TEAM_ABBREVIATION",
    "SEASON",

    # direct outcome leakage
    "W",
    "L",
    "PLUS_MINUS",

    # target itself
    "W_PCT",

    # low-value / redundant basics
    "MIN",
    "GP",
    "G",

    # optional: remove raw scoring totals if you want cleaner feature selection
    "PTS",
    "FGM",
    "FGA",
    "FTM",
    "FTA",
    "OREB",
    "DREB",
}

drop_patterns = [
    "_RANK"
]

cols_to_drop = [
    col for col in df_all.columns
    if (
        col in drop_exact
        or any(pat in col for pat in drop_patterns)
    )
]

Fetching 2021-22...


NameError: name 'leaguedashteamstats' is not defined

In [4]:
# Build modeling dataframe
y = df_all[target]

X = df_all.drop(columns=cols_to_drop)

# numerical features only
X = X.select_dtypes(include="number")

df_model = pd.concat([y, X], axis=1).dropna()

# -----------------------------
# TARGET + FEATURES
# -----------------------------

y = df_model["W_PCT"]
X = df_model.drop(columns=["W_PCT"])

# -----------------------------
# SCALE FEATURES
# -----------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# ELASTIC NET REGRESSION
# -----------------------------

model = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    random_state=42,
    max_iter=100000
)

model.fit(X_scaled, y)

# -----------------------------
# FEATURE IMPORTANCE
# -----------------------------

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coef_df["Abs_Coeff"] = coef_df["Coefficient"].abs()

important_features = (
    coef_df[
        (coef_df["Coefficient"] != 0)
        & (coef_df["Abs_Coeff"] >= 0.01)
    ]
    .sort_values("Abs_Coeff", ascending=False)
)

important_features["Coefficient"] = important_features["Coefficient"].round(3)

print(
    important_features[
        ["Feature", "Coefficient"]
    ]
)

    Feature  Coefficient
5       REB        0.054
0    FG_PCT        0.048
8       STL        0.041
7       TOV       -0.040
10     BLKA       -0.034
3   FG3_PCT        0.033
12      PFD        0.022
1      FG3M        0.015
6       AST       -0.015
4    FT_PCT        0.013


Ignoring assists, I can build a model from here.

In [5]:
features = [
    "FG_PCT",
    "FG3M",
    "FG3_PCT",
    "FT_PCT",
    "REB",
    "TOV",
    "STL",
    "BLKA",
    "PFD"
]

new_team = pd.DataFrame([{
    "FG_PCT": 0.485,
    "FG3M": 13.2,
    "FG3A": 35.5,
    "FG3_PCT": 0.372,
    "FT_PCT": 0.791,
    "REB": 44.1,
    "AST": 26.4,
    "TOV": 12.8,
    "STL": 7.9,
    "BLK": 5.1,
    "BLKA": 4.3,
    "PF": 18.2,
    "PFD": 19.7
}])

# force same column order as training
new_team = new_team[X.columns]

new_team_scaled = scaler.transform(new_team)

predicted_w_pct = model.predict(new_team_scaled)

print(predicted_w_pct)

[0.67371795]
